Replaced openpyxl with stdlib zipfile/XML parsing to fix ModuleNotFoundError in environment without external access.
*Co-authored with CoCo*

# 01 Load Excel Files

* Author: Jeremiah Hansen
* Last Updated: 6/17/2026

This notebook will load data into the `LOCATION` and `ORDER_DETAIL` tables from Excel files.

This currently does not use Snowpark File Access as it doesn't yet work in Notebooks. So for now we copy the file locally first.

In [ ]:
# Import python packages
import sys
import logging

# Set up the logger
logger_name = 'demo_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

# Set default values for debugging
notebook_name = '01_load_excel_files.ipynb'
database_name = 'DEMO_DB'
schema_name = 'DEV_SCHEMA'
role_name = 'DEMO_ROLE'

# Override values with passed notebook arguments
if sys.argv[0].endswith('.ipynb'):
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--database-name', type=str)
    parser.add_argument('--schema-name', type=str)
    parser.add_argument('--role-name', type=str)
    args, args_unknown = parser.parse_known_args()

    notebook_name = parser.prog  # same as argv[0]
    database_name = args.database_name or database_name
    schema_name = args.schema_name or schema_name
    role_name = args.role_name or role_name

# Get a Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set the default database and schema for the following cells
session.use_schema(f"{database_name}.{schema_name}")

# Set the role
# Needed when running EXECUTE NOTEBOOK PROJECT directly (since it ignores session context and uses the user's default role)
session.use_role(role_name)

# Get details about the current state
current_state_df = session.sql(f"""
        SELECT OBJECT_CONSTRUCT(
            'current_user', CURRENT_USER(),
            'current_role', CURRENT_ROLE(),
            'current_secondary_roles', PARSE_JSON(CURRENT_SECONDARY_ROLES()),
            'current_database', CURRENT_DATABASE(),
            'current_schema', CURRENT_SCHEMA()
        )::STRING AS session_context;
    """).collect()

logger.info(f"Begin executing notebook {notebook_name}", extra = {'logger_name': logger_name})
logger.info(f"Using parameters database: {database_name}, schema: {schema_name}, role: {role_name}", extra = {'logger_name': logger_name})
logger.info(f"Using session context {current_state_df[0]['SESSION_CONTEXT']}", extra = {'logger_name': logger_name})

In [ ]:
!pip install openpyxl

In [ ]:
%%sql -r dataframe_1
-- Temporary solution to load in the metadata, this should be replaced with a directy query to a directory table (or a metadata table)
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/order_detail.xlsx' AS STAGE_FILE_PATH, 'order_detail' AS WORKSHEET_NAME, 'ORDER_DETAIL' AS TARGET_TABLE
UNION
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/location.xlsx', 'location', 'LOCATION';

## Create a function to load Excel worksheet to table

Create a reusable function to load an Excel worksheet to a table in Snowflake.

Note: Until we can use scoped URLs in Notebooks, via the `BUILD_SCOPED_FILE_URL()` function, we need to temporarily copy the file to a temp stage and then process from there.

In [ ]:
from snowflake.snowpark.files import SnowflakeFile
import pandas as pd
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import re

# 1. Create a temp internal stage (once at the start)
session.sql("CREATE TEMP STAGE IF NOT EXISTS temp_excel_stage").collect()

def _col_index(col_ref):
    """Convert Excel column reference (e.g. 'A', 'AB') to 0-based index."""
    letters = re.match(r'[A-Z]+', col_ref).group()
    idx = 0
    for ch in letters:
        idx = idx * 26 + (ord(ch) - ord('A') + 1)
    return idx - 1

def parse_xlsx_sheet(file_bytes, worksheet_name):
    """Parse an xlsx worksheet using stdlib zipfile and xml.etree (no openpyxl needed)"""
    with zipfile.ZipFile(BytesIO(file_bytes)) as zf:
        # Read shared strings
        shared_strings = []
        if 'xl/sharedStrings.xml' in zf.namelist():
            ss_tree = ET.parse(zf.open('xl/sharedStrings.xml'))
            ns = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            for si in ss_tree.findall('.//s:si', ns):
                text_parts = [t.text or '' for t in si.findall('.//s:t', ns)]
                shared_strings.append(''.join(text_parts))

        # Find sheet by name from workbook.xml
        wb_tree = ET.parse(zf.open('xl/workbook.xml'))
        ns_wb = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
                 'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'}
        sheet_elem = None
        for s in wb_tree.findall('.//s:sheet', ns_wb):
            if s.get('name') == worksheet_name:
                sheet_elem = s
                break
        if sheet_elem is None:
            raise ValueError(f"Worksheet '{worksheet_name}' not found")

        # Get sheet relationship ID and resolve to file path
        r_id = sheet_elem.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
        rels_tree = ET.parse(zf.open('xl/_rels/workbook.xml.rels'))
        ns_rel = {'r': 'http://schemas.openxmlformats.org/package/2006/relationships'}
        sheet_path = None
        for rel in rels_tree.findall('.//r:Relationship', ns_rel):
            if rel.get('Id') == r_id:
                sheet_path = 'xl/' + rel.get('Target')
                break

        # Parse the sheet XML
        sheet_tree = ET.parse(zf.open(sheet_path))
        ns_s = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

        # First pass: determine max column count from all rows
        all_rows = sheet_tree.findall('.//s:sheetData/s:row', ns_s)
        max_cols = 0
        for row in all_rows:
            for cell in row.findall('s:c', ns_s):
                ref = cell.get('r')
                if ref:
                    col_idx = _col_index(ref)
                    if col_idx + 1 > max_cols:
                        max_cols = col_idx + 1

        # Second pass: build rows with positional placement
        rows = []
        for row in all_rows:
            row_data = [None] * max_cols
            for cell in row.findall('s:c', ns_s):
                ref = cell.get('r')
                col_idx = _col_index(ref) if ref else len([c for c in row_data if c is not None])
                cell_type = cell.get('t')
                val_elem = cell.find('s:v', ns_s)
                if val_elem is None or val_elem.text is None:
                    value = None
                elif cell_type == 's':
                    value = shared_strings[int(val_elem.text)]
                elif cell_type == 'b':
                    value = bool(int(val_elem.text))
                else:
                    try:
                        value = float(val_elem.text) if '.' in val_elem.text else int(val_elem.text)
                    except ValueError:
                        value = val_elem.text
                row_data[col_idx] = value
            rows.append(row_data)

    # First row is header
    columns = rows[0]
    data = rows[1:]
    return pd.DataFrame(data, columns=columns)

def load_excel_worksheet_to_table(session, external_path, worksheet_name, target_table):
    """Load an Excel worksheet by copying to internal stage first"""
    
    # Extract filename from path
    filename = external_path.split('/')[-1]
    
    # 2. Copy file from external to internal stage
    session.sql(f"""
        COPY FILES INTO @temp_excel_stage
        FROM {external_path}
    """).collect()
    
    # 3. Read file from internal stage and parse
    with SnowflakeFile.open(f'@temp_excel_stage/{filename}', 'rb') as f:
        file_bytes = f.read()
    
    df = parse_xlsx_sheet(file_bytes, worksheet_name)
    
    # Write to Snowflake table
    snowpark_df = session.create_dataframe(df)
    snowpark_df.write.mode("overwrite").save_as_table(target_table)
    
    logger.info(f"Loaded {len(df)} rows from '{worksheet_name}' to {target_table}", extra = {'logger_name': logger_name})

## Process all Excel worksheets

Loop through each Excel worksheet to process and call our `load_excel_worksheet_to_table_local()` function.

In [ ]:
# Process each file from the sql_get_spreadsheets cell above
files_to_load = dataframe_1.collect()
for excel_file in files_to_load:
    print(f"Processing Excel file {excel_file['STAGE_FILE_PATH']}")
    load_excel_worksheet_to_table(session, excel_file['STAGE_FILE_PATH'], excel_file['WORKSHEET_NAME'], excel_file['TARGET_TABLE'])

logger.info(f"Finish executing notebook {notebook_name}", extra = {'logger_name': logger_name})

### Debugging

In [ ]:
%%sql -r dataframe_2
--DESCRIBE TABLE LOCATION;
--SELECT * FROM LOCATION;
--SHOW TABLES;